# Motion Detection Simulator

This notebook runs an interactive motion-detection pipeline over image-sequence folders. It lets you choose the dataset, spatial smoothing strategy, temporal derivative operator, and thresholding method, then displays and exports each processing stage.

## Setup

Install the project dependencies before running the notebook:

```bash
python -m pip install -r requirements.txt
```

## Data

By default, the notebook looks for image sequences under `Database/` in the project root. You can also change the database path from the dashboard. macOS metadata folders such as `__MACOSX` are ignored automatically.

## Output

Each run is saved under `Kabhilesh-Aidan-Project-Output/` with separate folders for original frames, grayscale frames, smoothing output, temporal derivatives, threshold masks, and final detected-motion overlays.


In [1]:
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "cv2": "opencv-python",
    "ipywidgets": "ipywidgets",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "scipy": "scipy",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        *missing_packages,
    ])

from IPython.display import display, clear_output

import src.gui as gui_mod
import src.Utilities as utilities_mod
import src.PostProcess as post_mod

# Force reload so notebook uses latest function signatures
importlib.reload(gui_mod)
importlib.reload(utilities_mod)
importlib.reload(post_mod)

from src.VideoReader import ImageSequenceReader
from src.SpatialSmoothing import SpatialSmoothing, BoxFilter, Gaussian2DFilter
from src.TemporalDerivative import Temporal_Derivative, SimpleTemporalDifference, OneDCenteredDifference, OneDDerivativeOfGaussian
from src.Threshold import Threshold, ThresholdStrategy, ManualThreshold, AdaptiveThreshold

# This input can be changed in the GUI, but this local default matches this project folder.
DEFAULT_DB_ROOT = str(Path.cwd() / "Database")
if not Path(DEFAULT_DB_ROOT).exists():
    DEFAULT_DB_ROOT = r"F:\ML\CV_Project1_Codebase\Database"

build_gui = gui_mod.build_gui
play_side_by_side = utilities_mod.play_side_by_side
deriv_to_u8 = utilities_mod.deriv_to_u8


def run_pipeline_and_play(cfg, out):

    # ---------- 1) Read frames and Convert to grayscale ----------
    reader = ImageSequenceReader(cfg["video_path"])
    orig_frames, gray_frames = reader.frames_bgr_and_gray()

    # ---------- 2) Smoothing ----------
    s_mode = cfg["smoothing"]["mode"]
    temporal_input_frames = gray_frames

    if s_mode == "box3":
        smoothing_name = "box3"
        temporal_input_frames = SpatialSmoothing(BoxFilter(kernel_size=3)).process(gray_frames)
    elif s_mode == "box5":
        smoothing_name = "box5"
        temporal_input_frames = SpatialSmoothing(BoxFilter(kernel_size=5)).process(gray_frames)
    elif s_mode == "sigma":
        s_sigma = cfg["smoothing"]["smoothing_sigma"]
        smoothing_name = f"gaussian_sigma_{s_sigma:g}"
        temporal_input_frames = SpatialSmoothing(Gaussian2DFilter(sigma=s_sigma)).process(gray_frames)
    else:
        smoothing_name = "none"

    # ---------- 3) Temporal derivative ----------
    td_mode = cfg["temporal"]["mode"]

    def make_td_strategy(mode, sigma=None, k=3):
        if mode in ("op1d", "simple"):
            return SimpleTemporalDifference()
        if mode == "centered":
            return OneDCenteredDifference()
        if mode == "dog":
            if sigma is None:
                raise ValueError("temporal_sigma is required for DoG mode")
            return OneDDerivativeOfGaussian(sigma=sigma, k=k)
        raise ValueError(f"Unknown temporal mode: {mode}")

    td_strategy = make_td_strategy(td_mode, sigma=cfg["temporal"].get("temporal_sigma"))
    td = Temporal_Derivative(td_strategy)

    idx_list = []
    deriv_list = []
    for idx, d in td.apply(enumerate(temporal_input_frames)):
        idx_list.append(int(idx))
        deriv_list.append(d)

    if not deriv_list:
        with out:
            clear_output(wait=True)
            print("No temporal derivative frames produced for this configuration.")
        return

    if td_mode == "dog":
        td_name = f"dog_sigma_{cfg['temporal'].get('temporal_sigma'):g}"
    else:
        td_name = td_mode

    # ---------- 4) Threshold ----------
    t_mode = cfg["threshold"]["mode"]

    def make_threshold_strategy(mode: str) -> ThresholdStrategy:
        if mode == "manual":
            t_value = int(cfg["threshold"]["threshold_value"])
            return ManualThreshold(threshold_value=t_value)
        if mode == "adaptive":
            return AdaptiveThreshold()
        raise ValueError(f"Unknown threshold mode: {mode}")

    thr = Threshold(make_threshold_strategy(t_mode))
    mask_by_idx = {}
    for idx, m in thr.apply(zip(idx_list, deriv_list)):
        mask_by_idx[int(idx)] = m

    if not mask_by_idx:
        with out:
            clear_output(wait=True)
            print("No threshold frames produced for this configuration.")
        return

    if t_mode == "manual":
        thr_name = f"manual_{int(cfg['threshold']['threshold_value'])}"
    else:
        thr_name = "adaptive"

    # ---------- 5) Build aligned streams ----------
    aligned_indices, aligned_deriv, aligned_mask, aligned_overlay = post_mod.build_aligned_streams(orig_frames, idx_list, deriv_list, mask_by_idx)

    # ---------- 6) Save outputs in organized folders ----------
    run_dir = post_mod.save_run_outputs(cfg, smoothing_name, td_name, thr_name, orig_frames, gray_frames, 
    temporal_input_frames, aligned_indices, aligned_deriv, aligned_mask, aligned_overlay)

    # ---------- 7) Playback views ----------
    frame_iter = post_mod.six_view_generator(orig_frames, gray_frames, temporal_input_frames, idx_list, deriv_list, mask_by_idx, deriv_to_u8)

    with out:
        print(f"Saved run outputs to: {run_dir}")


    play_side_by_side(frame_iter, fps=30, figsize=(21, 4), out=out)

try:
    VIDEO_OPTIONS = utilities_mod.list_sequence_folders(DEFAULT_DB_ROOT)
except Exception:
    VIDEO_OPTIONS = []

ui, get_config, wait_for_run, out = build_gui(VIDEO_OPTIONS, default_database_root=DEFAULT_DB_ROOT, on_run=run_pipeline_and_play)
display(ui)